In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
%run /Users/anuragsaha1712@gmail.com/consolidated_pipeline/FCMG_DB/1_setup/utilities

In [0]:
print(bronze_schema)

In [0]:
dbutils.widgets.text("catalog","fcmg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar-anu/{data_source}/*.csv"
print(base_path)


In [0]:
df = (spark.read.format("csv")
    .option("header",True)
    .option("inferSchema",True)
    .load(base_path)
    .withColumn("read_timestamp",f.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
    )

In [0]:
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}")

df.write\
    .format("delta") \
    .option("delta.enableChangeDataFeed","true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter("count > 1")
df_duplicates.show()


In [0]:
df_silver = df_bronze.dropDuplicates(["customer_id"])
df_silver.show()

In [0]:
print("before",df_bronze.count())
print("after",df_silver.count())

In [0]:
display(df_silver.filter(f.col("customer_name") != f.trim(f.col("customer_name"))))

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    f.trim(f.col("customer_name"))
)
display(df_silver.filter(f.col("customer_name") != f.trim(f.col("customer_name"))))
df_silver = df_silver.withColumn(
    "customer_name",
    f.initcap(f.col("customer_name"))
)
display(df_silver.filter(f.col("customer_name") != f.trim(f.col("customer_name"))))


In [0]:
df_silver.select("city").distinct().show()

In [0]:
city_mapping = {
    'hyderabadd' : 'hyderabad',
    'Bengalore' : 'Bengaluru',
    'Bengaluruu' : 'Bengaluru',
    'Hyderbad' : 'Hyderabad',

    'NewDelhi' : 'New Delhi',
    'NewDheli' : 'New Delhi',
    'NewDelhee' : 'New Delhi'
}

allowed = ["Bengaluru","Hyderabad","New Delhi"]
df_silver = (
    df_silver
    .replace(city_mapping , subset=["city"])
    .withColumn(
        "city",
        f.when(f.col("city").isNull(),None)
        .when(f.col("city").isin(allowed) , f.col("city"))
        .otherwise(None)
    )
)

In [0]:
df_silver.select("city").distinct().show()


In [0]:
df_silver.select("customer_name").distinct().show() 

In [0]:
df_silver.filter(f.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ["Macrobite Superfoods" , "Sprintx Nutrition" , "Zenathlete Foods" , "Primefuel Nutrition" , "Recovery Lane"]
df_silver.filter(f.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
customer_city_fix = {
    789221 : 'New Delhi',
    789403 : 'New Delhi',
    789420 : 'Bengaluru',
    789521 : 'Hyderabad',
    789603 : 'Hyderabad'
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id","fixed_city"]
)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, on="customer_id", how="left")
    .withColumn(
        "city",
        f.coalesce(f.col("city"), f.col("fixed_city"))
    )
    .drop("fixed_city")
)

display(df_silver)

In [0]:
df_silver = df_silver.withColumn("customer_id", f.col("customer_id").cast("string"))


In [0]:
df_silver.printSchema()